# net模式

In [ ]:
# 将模型设置为评估模式
net.eval()

# 将模型设置为训练模式
net.train()

# -------------------------------如何使用的例子
if isinstance(net, torch.nn.Module):
    net.train()

In [ ]:
# 正态分布n(0,1), size (100, 10)
torch.normal(0, 1, (100, 10))

# data_iter的分类

In [ ]:
# -------------------------------如何生成训练集和测试集的迭代器
from torch.utils import data
from torchvision import transforms
# 此处是为了下载mnist数据集
import torchvision


def load_data_fashion_mnist(batch_size, resize=None):
    # 通过ToTensor实例将图像数据从PIL类型变换成32位浮点数格式，并除以255使得所有像素的数值均在0到1之间
    trans = [transforms.ToTensor()]
    # 尺寸裁剪
    if resize:
        trans.insert(0, transforms.Resize(resize))
    trans = transforms.Compose(trans)

    # 下载Fashion-MNIST数据集
    mnist_train = torchvision.datasets.FashionMNIST(
        root="../data", train=True, transform=trans, download=True)
    mnist_test = torchvision.datasets.FashionMNIST(
        root="../data", train=False, transform=trans, download=True)

    # 将其加载到内存中
    # num_workers, 使用4个进程来读取数据
    train_iter = data.DataLoader(mnist_train, batch_size,
                                 shuffle=True, num_workers=4)
    test_iter = data.DataLoader(mnist_test, batch_size,
                                shuffle=False, num_workers=4)
    returen train_iter, test_iter


train_iter, test_iter = load_data_fashion_mnist(batch_size)


# -------------------------------如何遍历的例子
for X, y in data_iter:
    metric.add(accuracy(net(X), y), y.numel())

# 准备

## 初始化权重, 优化器, 学习率

> 方法1

In [ ]:
# 以均值0和标准差0.01随机初始化权重。
def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)

net.apply(init_weights);

# 学习率为0.1的小批量随机梯度下降作为优化算法
optimizer = torch.optim.SGD(net.parameters(), lr=0.1)

> 方法2

In [ ]:
num_inputs, num_outputs, num_hiddens = 784, 10, 256

W1 = nn.Parameter(torch.randn(
    num_inputs, num_hiddens, requires_grad=True) * 0.01)
b1 = nn.Parameter(torch.zeros(num_hiddens, requires_grad=True))
W2 = nn.Parameter(torch.randn(
    num_hiddens, num_outputs, requires_grad=True) * 0.01)
b2 = nn.Parameter(torch.zeros(num_outputs, requires_grad=True))

params = [W1, b1, W2, b2]

optimizer = torch.optim.SGD(params, lr=lr)

> 权重衰减


默认情况下，PyTorch同时衰减权重和偏移。

直接通过weight_decay指定weight decay超参数。这里我们只为权重设置了weight_decay，所以偏置参数不会衰减。

In [ ]:
# 偏置参数没有衰减
optimizer = torch.optim.SGD(
    [
        {"params": net[0].weight, 'weight_decay': wd},
        {"params": net[0].bias}
    ],
    lr=lr
)

> 其他

In [ ]:
torch.optim.SGD()

# Adam优化器的主要吸引力在于它对初始学习率不那么敏感。
torch.optim.Adam()

## loss

In [ ]:
# 交叉熵损失
loss = nn.CrossEntropyLoss(reduction='none')

# 均方误差
loss = nn.MSELoss(reduction='none')

# train

In [ ]:
import malplotlib.plot as plt


# -------------------------------计算预测正确的数量
def accuracy(y_hat, y):
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1)
    cmp = y_hat.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum())


# -------------------------------在n个变量上累加
class Accumulator:

    def __init__(self, n):
        self.data = [0.0] * n

    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]
    pass


# -------------------------------训练模型一个迭代周期
def train_epoch_ch3(net, train_iter, loss, trainer):

    # 将模型设置为训练模式
    if isinstance(net, torch.nn.Module):
        net.train()
    # 训练损失总和、训练准确度总和、样本数
    metric = Accumulator(3)
    for X, y in train_iter:
        # 计算梯度并更新参数
        y_hat = net(X)
        l = loss(y_hat, y)
        if isinstance(updater, torch.optim.Optimizer):
            # 使用PyTorch内置的优化器和损失函数
            updater.zero_grad()
            l.mean().backward()
            updater.step()
        else:
            # 使用定制的优化器和损失函数
            l.sum().backward()
            updater(X.shape[0])
        metric.add(float(l.sum()), accuracy(y_hat, y), y.numel())
    # 返回训练损失和训练精度
    return metric[0] / metric[2], metric[1] / metric[2]


# -------------------------------一个在动画中绘制数据的实用程序类Animator
class Animator:
    def __init__(self, xlabel=None, ylabel=None, legend=None, xlim=None,
                 ylim=None, xscale='linear', yscale='linear',
                 fmts=('-', 'm--', 'g-.', 'r:'), nrows=1, ncols=1,
                 figsize=(3.5, 2.5)):
        # 增量地绘制多条线
        if legend is None:
            legend = []
        self.fig, self.axes = plt.subplots(nrows, ncols, figsize=figsize)
        if nrows * ncols == 1:
            self.axes = [self.axes, ]
        # 使用lambda函数捕获参数
        self.config_axes = lambda: set_axes(
            self.axes[0], xlabel, ylabel, xlim, ylim, xscale, yscale, legend)
        self.X, self.Y, self.fmts = None, None, fmts

    def add(self, x, y):
        # 向图表中添加多个数据点
        if not hasattr(y, "__len__"):
            y = [y]
        n = len(y)
        if not hasattr(x, "__len__"):
            x = [x] * n
        if not self.X:
            self.X = [[] for _ in range(n)]
        if not self.Y:
            self.Y = [[] for _ in range(n)]
        for i, (a, b) in enumerate(zip(x, y)):
            if a is not None and b is not None:
                self.X[i].append(a)
                self.Y[i].append(b)
        self.axes[0].cla()
        for x, y, fmt in zip(self.X, self.Y, self.fmts):
            self.axes[0].plot(x, y, fmt)
        self.config_axes()
        display.display(self.fig)
        display.clear_output(wait=True)


# -------------------------------训练模型
def train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer):
    animator = Animator(xlabel='epoch', xlim=[1, num_epochs], ylim=[0.3, 0.9],
                        legend=['train loss', 'train acc', 'test acc'])
    for epoch in range(num_epochs):
        train_metrics = train_epoch_ch3(net, train_iter, loss, trainer)
        test_acc = evaluate_accuracy(net, test_iter)
        animator.add(epoch + 1, train_metrics + (test_acc,))
    train_loss, train_acc = train_metrics
    assert train_loss < 0.5, train_loss
    assert train_acc <= 1 and train_acc > 0.7, train_acc
    assert test_acc <= 1 and test_acc > 0.7, test_acc

# 激活函数

## 修正线性单元（Rectified linear unit，ReLU）

$$\operatorname{ReLU}(x) = \max(x, 0)$$.

求导表现得特别好(要么是0,要么是1)：要么让参数消失，要么让参数通过

0点不可导, 不光滑

## sigmoid 挤压函数（squashing function）

$$\operatorname{sigmoid}(x) = \frac{1}{1 + \exp(-x)}$$.

$$\frac{d}{dx} \operatorname{sigmoid}(x) = \frac{\exp(-x)}{(1 + \exp(-x))^2} = \operatorname{sigmoid}(x)\left(1-\operatorname{sigmoid}(x)\right)$$.

它将范围（-inf, inf）中的任意输入压缩到区间（0, 1）中的某个值.

平滑的、可微的.

当输入为0时，sigmoid函数的导数达到最大值0.25； 而输入在任一方向上越远离0点时，导数越接近0。



## tanh函数
$$\operatorname{tanh}(x) = \frac{1 - \exp(-2x)}{1 + \exp(-2x)}$$.

将其输入压缩转换到区间(-1, 1)

不同的是tanh函数关于坐标系原点中心对称。

$$\frac{d}{dx} \operatorname{tanh}(x) = 1 - \operatorname{tanh}^2(x)$$.

当输入接近0时，tanh函数的导数接近最大值1。 与我们在sigmoid函数图像中看到的类似， 输入在任一方向上越远离0点，导数越接近0。

# nn

In [5]:
import torch
from torch import nn
from torch.nn import functional as F

# 顺序块
## 官方Sequential

In [6]:
net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
X = torch.rand(2, 20)
net(X)

tensor([[-0.0036, -0.2143, -0.2751,  0.0393, -0.0237, -0.0586, -0.0943, -0.0497,
          0.1755,  0.2041],
        [-0.0477, -0.1515, -0.3373,  0.0158, -0.0297, -0.1507, -0.0481, -0.1468,
          0.1065,  0.1961]], grad_fn=<AddmmBackward0>)

## 自定义Sequential

In [ ]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # 这里，module是Module子类的一个实例。我们把它保存在'Module'类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module

    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
X = torch.rand(2, 20)
net(X)

## 自定义块

In [ ]:
class MLP(nn.Module):
    # 用模型参数声明层。这里，我们声明两个全连接的层
    def __init__(self):
        # 调用MLP的父类Module的构造函数来执行必要的初始化。
        # 这样，在类实例化时也可以指定其他函数参数，例如模型参数params（稍后将介绍）
        super().__init__()
        self.hidden = nn.Linear(20, 256)  # 隐藏层
        self.out = nn.Linear(256, 10)  # 输出层

    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self, X):
        # 注意，这里我们使用ReLU的函数版本，其在nn.functional模块中定义。
        return self.out(F.relu(self.hidden(X)))

In [ ]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数。因此其在训练期间保持不变
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 复用全连接层。这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

net = FixedHiddenMLP()
X = torch.rand(2, 20)
net(X)

In [ ]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
X = torch.rand(2, 20)
chimera(X)

## nn

In [ ]:
nn.ReLU()

nn.Dropout(0.1)

nn.Flatten()

nn.Linear(256, 10)